<div align="center">

<a href="https://colab.research.google.com/github/utkukose/machine-ethics-ai-safety-NB-lecture/blob/main/exams/midterm/MID2_triage_decision_procedures.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Midterm Capstone: Ethics, Fairness and Specification in Practice

## Project 2: Ethical theories as allocation procedures in intensive care triage

**Machine Ethics and Artificial Intelligence Safety (11118BLG001)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## The project

Scarce intensive care beds are allocated by six procedures that encode different moral commitments: two utilitarian rules, first come first served, a lottery among patients who would benefit, a prioritarian rule and a hybrid with a futility rule [1, 2]. The example evaluates the procedures on simulated cohorts, aggregates the rankings of five stakeholder groups with Borda and Condorcet methods and recommends a policy under moral uncertainty [3, 4].

**Required extensions.** Add at least one deontological constraint, such as a ban on age as a criterion, and one procedural value, such as appeals or re-evaluation over time, and show how they change outcomes. Study the sensitivity of the recommendation to the credences and to the normalisation of choiceworthiness, test for Condorcet cycles with more stakeholder profiles, and discuss which parts of the decision should never be automated.

## What this example implements

The notebook simulates cohorts of patients, evaluates six allocation procedures on lives saved, life-years saved, group shares and service to the worst-off, aggregates stakeholder rankings and applies maximisation of expected choiceworthiness.

Run the cells in order. The example is a baseline: the capstone extends it, evaluates the extensions and reports the results.

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Cohorts and allocation procedures

In [ ]:
import pandas as pd
rng = np.random.default_rng(SEED)
AGE = {"young": (0.25, 45), "middle": (0.45, 25), "older": (0.30, 10)}   # population share and remaining life-years if the patient survives

def cohort(n=40):
    groups = rng.choice(list(AGE), n, p=[v[0] for v in AGE.values()])
    p_no = np.clip(rng.beta(2, 3, n) - (groups == "older") * 0.1, 0.01, 0.95)   # survival without intensive care
    p_icu = np.clip(p_no + rng.beta(2, 4, n) * 0.6, 0, 0.99)                      # survival with intensive care
    life_years = np.array([AGE[g][1] for g in groups]) * rng.uniform(0.8, 1.2, n)
    return pd.DataFrame({"group": groups, "p_no": p_no, "p_icu": p_icu, "life_years": life_years, "arrival": rng.permutation(n)})

def benefit(df):
    return df.p_icu - df.p_no

POLICIES = {
    "utilitarian: lives": lambda df, K: benefit(df).nlargest(K).index,
    "utilitarian: life-years": lambda df, K: (benefit(df) * df.life_years).nlargest(K).index,
    "first come, first served": lambda df, K: df.arrival.nsmallest(K).index,
    "lottery among non-futile": lambda df, K: rng.choice(df.index[benefit(df) > 0.05], min(K, int((benefit(df) > 0.05).sum())), replace=False),
    "prioritarian": lambda df, K: (benefit(df) * (1 - df.p_no) ** 2).nlargest(K).index,
    "hybrid: futility rule, then lives": lambda df, K: benefit(df)[benefit(df) > 0.05].nlargest(K).index,
}

def evaluate(policy, days=300, K=10):
    rows = []
    for _ in range(days):
        df = cohort()
        chosen = pd.Index(policy(df, K))
        b = benefit(df)
        rows.append({"lives saved": b[chosen].sum(), "life-years saved": (b * df.life_years)[chosen].sum(),
                     **{f"share {g}": np.mean(df.loc[chosen, "group"] == g) for g in AGE},
                     "worst-off served": np.mean(df.loc[chosen, "p_no"] < df.p_no.quantile(0.25))})
    return pd.DataFrame(rows).mean()

table = pd.DataFrame({name: evaluate(f) for name, f in POLICIES.items()}).T
table.round(3)

## 2. Stakeholder aggregation

In [ ]:
STAKEHOLDERS = {
    "intensivists": {"lives saved": 1.0, "life-years saved": 0.3},
    "patient advocates": {"worst-off served": 1.0, "lives saved": 0.3},
    "older citizens' association": {"share older": 1.0},
    "health economists": {"life-years saved": 1.0},
    "ethics committee": {"lives saved": 0.6, "worst-off served": 0.6},
}
z = (table - table.mean()) / table.std(ddof=0)
rankings = {s: (z[list(w)] * pd.Series(w)).sum(axis=1).sort_values(ascending=False).index.tolist() for s, w in STAKEHOLDERS.items()}

def borda(rk):
    score = {p: 0 for p in table.index}
    for r in rk.values():
        for pos, p in enumerate(r):
            score[p] += len(r) - 1 - pos
    return pd.Series(score).sort_values(ascending=False)

def condorcet(rk):
    for a in table.index:
        if all(sum(r.index(a) < r.index(b) for r in rk.values()) > len(rk) / 2 for b in table.index if b != a):
            return a
    return None

print("Borda ranking:\n", borda(rankings))
print("Condorcet winner:", condorcet(rankings))

## 3. Deciding under moral uncertainty

In [ ]:
THEORIES = {"utilitarian (lives)": "lives saved", "utilitarian (life-years)": "life-years saved", "prioritarian": "worst-off served"}
zt = (table[list(THEORIES.values())] - table[list(THEORIES.values())].mean()) / table[list(THEORIES.values())].std(ddof=0)

def mec(credences):
    c = pd.Series(credences, index=list(THEORIES.values()))
    return (zt * c).sum(axis=1).idxmax()

for cu in [0.8, 0.5, 0.2]:
    rest = (1 - cu) / 2
    print(f"credence in utilitarianism (lives) {cu}: recommended policy = {mec([cu, rest, rest])}")

## Report and submission

The technical report states the question, explains the design choices, presents the experiments with the figures produced by the notebook and discusses the ethical and safety implications and the limitations of the results. It cites at least eight sources in square brackets, at least four of them from the course readings, and links to the final Colab notebook.

This capstone also supports self-learning and can be completed at any pace. When the course is taught actively in a semester, the midterm capstone is sent by e-mail to utkukose@sdu.edu.tr or utkukose@gmail.com no later than 23:53 (Türkiye time) on the last Sunday of Week 7, with the report and all code files attached or linked.

## References

[1] Mill, J. S. (1863). *Utilitarianism*. Parker, Son, and Bourn.

[2] Rawls, J. (1971). *A Theory of Justice*. Harvard University Press.

[3] de Condorcet, M. (1785). *Essai sur l'application de l'analyse à la probabilité des décisions rendues à la pluralité des voix*. Imprimerie Royale.

[4] MacAskill, W., Bykvist, K., & Ord, T. (2020). *Moral Uncertainty*. Oxford University Press.